In [55]:
import tensorflow as ts
import pandas as pd # type: ignore
from IPython.display import display

In [39]:
class cell_state:
    def __init__(self, init_color, coord, grid_dim):
        color = init_color
        if color == "green":
            state_vector = [1, 0, 0]
        if color == "red":
            state_vector = [0, 1, 0]
        if color == "blue":
            state_vector = [0, 0, 1]
        
    def color_change():


In [50]:
import tkinter as tk
from random import randint
class cell_state:
    def __init__(self):
        pass

# Parameters
grid_size = 10
cell_size = 80  # Size of each cell in pixels

# Function to toggle colors in the grid
def toggle_color(event):
    x, y = event.x // cell_size, event.y // cell_size
    index = y * grid_size + x  # Calculate the index in the list
    
    current_color = colors[index]
    
    # Update color
    if current_color == "green":
        colors[index] = "red"
    elif current_color == "red":
        colors[index] = "blue"
    elif current_color == "blue":
        colors[index] = "green"

    # Change the color of the specific rectangle
    canvas.itemconfig(rect_ids[index], fill=colors[index])

# Parameters
grid_size = 10
cell_size = 80  # Size of each cell in pixels

# Create the main window
root = tk.Tk()
root.title("200x200 Grid")

# Create a Canvas
canvas = tk.Canvas(root, width=grid_size * cell_size, height=grid_size * cell_size)
canvas.pack()

# Initialize color list and rectangle IDs
colors = ["green"] * (grid_size * grid_size)
rect_ids = []

# Create the initial grid
for i in range(grid_size):
    for j in range(grid_size):
        x1, y1 = j * cell_size, i * cell_size
        x2, y2 = x1 + cell_size, y1 + cell_size
        rect_id = canvas.create_rectangle(x1, y1, x2, y2, fill="green", outline="")
        rect_ids.append(rect_id)

# Bind click event
canvas.bind("<Button-1>", toggle_color)



# Start the GUI event loop
root.mainloop()

In [37]:
import tkinter
from tkinter import ttk
from tkinter import NORMAL, DISABLED
from array import array


class LongMatrix:
    def __init__(self, width: int, height: int):
        self.data = array('L', (0 for i in range(width*height)))
        self.width = width
        self.height = height

    def _check_bounds(self, row: int, col: int):
        if not (0 <= row < self.height):
            raise IndexError(f'row index out of range for {self.width}x{self.height} matrix: {row}')

        if not (0 <= col < self.width):
            raise IndexError(f'column index out of range for {self.width}x{self.height} matrix: {col}')

    def __getitem__(self, row_col: tuple[int, int]) -> int:
        row = row_col[0]
        col = row_col[1]
        self._check_bounds(row, col)
        return self.data[row*self.width + col]

    def __setitem__(self, row_col: tuple[int, int], val: int):
        row = row_col[0]
        col = row_col[1]
        self._check_bounds(row, col)
        self.data[row*self.width + col] = val


class RepeatCommand:
    """Repeatedly call a function on a certain minimum interval without overwhelming the event loop."""
    def __init__(self,
                 root: tkinter.Tk,
                 interval_ms: int,
                 func,
                 *args,
                 **kwargs):
        self.root = root
        self.interval = str(interval_ms)
        self.idle_command = root.register(self.schedule_idle)
        self.interval_command = root.register(self.schedule_interval)
        self.handle = None
        self.func = func
        self.args = args
        self.kwargs = kwargs

    def start(self):
        self.handle = self.root.call('after', 'idle', self.interval_command)

    def cancel(self):
        self.root.call('after', 'cancel', self.handle)
        self.handle = None

    def schedule_idle(self):
        self.func(*self.args, **self.kwargs)
        self.handle = self.root.call('after', 'idle', self.interval_command)

    def schedule_interval(self):
        self.handle = self.root.call('after', self.interval, self.idle_command)


class GameOfLife:
    def __init__(self,
                 title: str = 'Game of Life',
                 pixel_size: tuple[int, int] = (1000, 500),
                 grid_size: tuple[int, int] = (50, 25),
                 step_ms: int = 250):
        self.step_ms = step_ms
        self.grid_size = grid_size
        self.root = tkinter.Tk()
        self.root.title(title)
        self.screen_width = self.root.winfo_screenwidth()
        self.screen_height = self.root.winfo_screenheight()
        # center the window
        self.root.geometry(
            f'+{(self.screen_width - pixel_size[0]) // 2}'
            f'+{(self.screen_height - pixel_size[1]) // 2}'
        )
        self.canvas_width = pixel_size[0]
        self.canvas_height = pixel_size[1]
        self.contents = ttk.Frame()
        self.contents.grid()

        self.canvas = tkinter.Canvas(self.root,
                                     width=self.canvas_width + 2,
                                     height=self.canvas_height + 2,
                                     highlightthickness=0)
        self.canvas.config(background='white')
        self.canvas.bind('<Button-1>', self.toggle_cell)
        self.canvas.grid(row=0, column=0, columnspan=3)

        self.step_btn = ttk.Button(text='Step once', command=self.step_once)
        self.step_btn.grid(row=1, column=0)

        self.start_stop_btn = ttk.Button(text='Start', command=self.start_stop)
        self.start_stop_btn.grid(row=1, column=1)
        self.running = False

        self.reset_btn = ttk.Button(text='Reset', command=self.reset, state=DISABLED)
        self.reset_btn.grid(row=1, column=2)

        # each cell of the game corresponds to a rectangle on the canvas.
        self.cell_width = pixel_size[0] / grid_size[0]
        self.cell_height = pixel_size[1] / grid_size[1]
        # The grid has three rows and columns off-screen on each side to ensure
        # visible patterns propagate correctly. Cells outside the grid
        # are always considered dead.
        self.cells = LongMatrix(grid_size[0] + 6, grid_size[1] + 6)
        id_bits = self.cells.data.itemsize * 8 - 2
        for y in range(grid_size[1] + 6):
            for x in range(grid_size[0] + 6):
                tkid = self.canvas.create_rectangle(
                    (x - 3)*self.cell_width + 1,
                    (y - 3)*self.cell_height + 1,
                    (x - 2)*self.cell_width + 1,
                    (y - 2)*self.cell_height + 1,
                    width=2,
                    fill='white',
                    outline='grey'
                )
                # bottom bit is current state,
                # next bit is next state,
                # the rest of the bits are the id
                if tkid.bit_length() > id_bits:
                    raise ValueError("canvas item id too big!")
                self.cells[y, x] = tkid << 2

        self.step_repeater = RepeatCommand(self.root, self.step_ms, self.step)

    def run(self):
        self.root.mainloop()

    def toggle_cell(self, e: tkinter.Event):
        row = int(e.y // self.cell_height) + 3
        col = int(e.x // self.cell_width) + 3
        cell = self.cells[row, col]
        if cell & 1:  # cell is alive
            cell &= ~1  # clear the bottom bit
            self.canvas.itemconfigure(cell >> 2, fill='white')
        else:
            cell |= 1  # set the bottom bit
            self.canvas.itemconfigure(cell >> 2, fill='black')
        self.cells[row, col] = cell

    def step_once(self):
        self.step()
        self.root.update_idletasks()

    def reset(self):
        for row in range(self.cells.height):
            for col in range(self.cells.width):
                self.cells[row, col] &= ~3  # clear the bottom two bits
                self.canvas.itemconfigure(self.cells[row, col] >> 2, fill='white')

    def start_stop(self):
        if self.running:
            self.start_stop_btn['text'] = 'Start'
            self.step_btn['state'] = NORMAL
            self.reset_btn['state'] = NORMAL
            self.step_repeater.cancel()
            self.running = False
        else:
            self.start_stop_btn['text'] = 'Stop'
            self.step_btn['state'] = DISABLED
            self.reset_btn['state'] = DISABLED
            self.step_repeater.start()
            self.running = True

    def step(self):
        print(self.cells.data)
        neighbor_offsets = [
            (-1, -1), (0, -1), (1, -1),
            (-1, 0), (1, 0),
            (-1, 1), (0, 1), (1, 1)
        ]
        for y in range(self.cells.height):
            for x in range(self.cells.width):
                neighbor_count = 0
                this_cell = self.cells[y, x]
                # count the number of live neighbors to this cell
                for x_offset, y_offset in neighbor_offsets:
                    neighbor_x = x + x_offset
                    neighbor_y = y + y_offset
                    # all neighbors that are off the grid are considered dead
                    if neighbor_x < 0 or neighbor_y < 0:
                        continue
                    if self.cells.width <= neighbor_x or self.cells.height <= neighbor_y:
                        continue
                    neighbor_count += self.cells[neighbor_y, neighbor_x] & 1

                if this_cell & 1:
                    if 2 <= neighbor_count <= 3:
                        this_cell |= 2  # set the second bit
                    else:
                        this_cell &= ~2  # clear the second bit
                else:
                    if neighbor_count == 4:
                        this_cell |= 2
                    else:
                        this_cell &= ~2

                if this_cell & 2:
                    self.canvas.itemconfigure(this_cell >> 2, fill='black')
                else:
                    self.canvas.itemconfigure(this_cell >> 2, fill='white')

                self.cells[y, x] = this_cell

        for row in range(self.cells.height):
            for col in range(self.cells.width):
                this_cell = self.cells[row, col]
                # this_cell & ~3 clears the bottom two bits
                self.cells[row, col] = (this_cell & ~3) | ((this_cell & 2) >> 1)


if __name__ == '__main__':
    GameOfLife().run()


array('L', [4, 8, 12, 16, 20, 24, 28, 32, 36, 40, 44, 48, 52, 56, 60, 64, 68, 72, 76, 80, 84, 88, 92, 96, 100, 104, 108, 112, 116, 120, 124, 128, 132, 136, 140, 144, 148, 152, 156, 160, 164, 168, 172, 176, 180, 184, 188, 192, 196, 200, 204, 208, 212, 216, 220, 224, 228, 232, 236, 240, 244, 248, 252, 256, 260, 264, 268, 272, 276, 280, 284, 288, 292, 296, 300, 304, 308, 312, 316, 320, 324, 328, 332, 336, 340, 344, 348, 352, 356, 360, 364, 368, 372, 376, 380, 384, 388, 392, 396, 400, 404, 408, 412, 416, 420, 424, 428, 432, 436, 440, 444, 448, 452, 456, 460, 464, 468, 472, 476, 480, 484, 488, 492, 496, 500, 504, 508, 512, 516, 520, 524, 528, 532, 536, 540, 544, 548, 552, 556, 560, 564, 568, 572, 576, 580, 584, 588, 592, 596, 600, 604, 608, 612, 616, 620, 624, 628, 632, 636, 640, 644, 648, 652, 656, 660, 664, 668, 672, 676, 680, 684, 688, 692, 696, 700, 704, 708, 712, 716, 720, 724, 728, 732, 736, 740, 744, 748, 752, 756, 760, 764, 768, 772, 776, 780, 784, 788, 792, 796, 800, 804, 808, 812,

In [74]:

ds = pd.DataFrame({"Col1":[1, 2, 3, 4, 5, 6], "Col2":[7, 8, 9, 10, 11, 12]})
test = ds["Col1"]
test[0] = 212
test = test.sort_values().reset_index()
test = test.set_index('index')
ds

,Col1,Col2
0,212,7
1,2,8
2,3,9
3,4,10
4,5,11
5,6,12
